# Smoke test: Colab TPU v5e + PSC L40S

The router runs here, next to vLLM-TPU. The L40S server runs as a PSC batch job; we reach it
through an SSH tunnel via the PSC login node (Colab can connect out, nothing can connect in).

```
Colab (TPU v5e)
 ├─ vLLM-TPU            127.0.0.1:8000   -> "tpu"  (slow device)
 ├─ ssh tunnel          127.0.0.1:8001 ──► bridges2.psc.edu ──► <L40S node>:8000   -> "gpu"  (fast device)
 └─ router/router.py    sends to both
```

**Before running:**
1. Runtime → Change runtime type → **TPU v5e**.
2. On PSC (login node, repo root): `sbatch scripts/psc/serve.sbatch`, then wait until
   `slurm-har-serve-<jobid>.out` shows `READY node=...`. Put that node name in the tunnel cell below.

In [ ]:
# 1. Repo + vLLM-TPU (skips the install if vllm already imports)
BRANCH = "main"   # or a branch you're testing, e.g. "yanze/colab-tpu-psc"
%cd /content
!test -d hetero-agent-routing || git clone -q https://github.com/clemrho/hetero-agent-routing
%cd /content/hetero-agent-routing
!git fetch -q && git checkout -q {BRANCH} && git pull -q
!git log --oneline -1
!python -c "import vllm" 2>/dev/null || pip install -q vllm-tpu
!pip install -q httpx
!python -c "import jax, vllm; print('vllm', vllm.__version__); print(jax.devices())"

In [ ]:
# 2. Start vLLM-TPU in the background (first start compiles with XLA: can take 10+ min)
import os, subprocess, time, urllib.request

REPO = "/content/hetero-agent-routing"
MODEL = "Qwen/Qwen3-0.6B"   # must match the PSC server
MAX_LEN = 8192
os.makedirs(f"{REPO}/logs", exist_ok=True)

def up(port):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=5)
        return True
    except Exception:
        return False

if not up(8000):
    tpu_log = open(f"{REPO}/logs/vllm_tpu.log", "w")
    tpu = subprocess.Popen(
        ["vllm", "serve", MODEL, "--port", "8000", "--max-model-len", str(MAX_LEN),
         "--enable-prefix-caching", "--enable-prompt-tokens-details"],
        stdout=tpu_log, stderr=subprocess.STDOUT)
    t0 = time.time()
    while not up(8000):
        if tpu.poll() is not None:
            print(open(f"{REPO}/logs/vllm_tpu.log").read()[-4000:])
            raise RuntimeError("vLLM-TPU exited, see the log above")
        print(f"waiting for vLLM-TPU... {time.time() - t0:.0f}s", end="\r")
        time.sleep(10)
print("vLLM-TPU ready on :8000")

In [ ]:
# 3. Check the TPU server: cold vs warm request, cached_tokens > 0 on warm
!python {REPO}/scripts/check_backend.py --url http://127.0.0.1:8000 --model {MODEL}

In [ ]:
# 4. SSH tunnel to the PSC L40S: 127.0.0.1:8001 -> <PSC_NODE>:<PSC_PORT>
PSC_USER = "your_psc_username"
PSC_NODE = "gl002"     # from the READY line in slurm-har-serve-<jobid>.out
PSC_PORT = 8000

import getpass
!apt-get -qq install -y sshpass > /dev/null
!pkill -f "ssh -f -N -o StrictHostKeyChecking" || true
os.environ["SSHPASS"] = getpass.getpass(f"PSC password for {PSC_USER}: ")
r = subprocess.run(
    f"sshpass -e ssh -f -N -o StrictHostKeyChecking=accept-new -o ServerAliveInterval=30 "
    f"-o ExitOnForwardFailure=yes -L 8001:{PSC_NODE}:{PSC_PORT} {PSC_USER}@bridges2.psc.edu",
    shell=True)
del os.environ["SSHPASS"]
for _ in range(10):
    if up(8001):
        break
    time.sleep(2)
print("tunnel up: L40S reachable on :8001" if up(8001) else
      "!! tunnel failed: check user/password/node, and that the PSC job is still running")

In [ ]:
# 5. Check the L40S server through the tunnel
!python {REPO}/scripts/check_backend.py --url http://127.0.0.1:8001 --model {MODEL}

In [ ]:
# 6. Router smoke test with synthetic workflows. LB is router.py's default policy.
#    "tpu" = Colab TPU v5e (slow), "gpu" = PSC L40S (fast).
POLICIES = ["LB"]          # all baselines: ["LB", "FF", "CA", "RR"]
N_WF = 4

run_dir = f"{REPO}/logs/colab_{int(time.time())}"   # fresh dir, so old runs don't mix in
os.makedirs(run_dir)
os.environ.update(TPU_URL="http://127.0.0.1:8000", GPU_URL="http://127.0.0.1:8001", MODEL=MODEL)
for p in POLICIES:
    !cd {run_dir} && python {REPO}/router/router.py --policy {p} --n_wf {N_WF}
!python {REPO}/analysis/analyze.py {run_dir}/logs/*.jsonl

**Reading the result:** `devices` shows how many calls went to each device. TTFT on `gpu` (L40S)
includes the network round trip Colab ⇄ PSC, so compare it with the `check_backend` numbers above.

When done: `scancel <jobid>` on PSC, and run the cell below to close the tunnel.

In [ ]:
!pkill -f "ssh -f -N -o StrictHostKeyChecking" && echo "tunnel closed" || echo "no tunnel running"